# BDS Real Data Trust Diagnostics

This notebook validates the current local BDS working tree against real test datasets from `tests/`. It focuses on the newest trust diagnostics: `bds doctor`, adapter `evidence_tier`, and ambiguous temperature-semantics metadata, while also printing current-sign, step/cycle, validation, and provenance details.

The notebook intentionally prints detailed per-file evidence so maintainers can inspect each dataset without opening separate reports.

In [1]:
from __future__ import annotations

import json
import sys
import time
import traceback
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find repository root")


repo_root = find_repo_root(Path.cwd().resolve())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import bds
from battery_data_standard.doctor import doctor

print("Repository root:", repo_root)
print("BDS version:", bds.__version__)
print("Notebook:", Path("examples/notebooks/bds_real_data_trust_diagnostics.ipynb"))
print("Current sign mode for real-data review: preserve")
print("Current sign sanity check: adjacent (expected to be inconclusive when sign is preserved)")

Repository root: C:\work\Batterydatastandard
BDS version: 0.2.2
Notebook: examples\notebooks\bds_real_data_trust_diagnostics.ipynb
Current sign mode for real-data review: preserve
Current sign sanity check: adjacent (expected to be inconclusive when sign is preserved)


## Runtime Format Evidence

This cell prints the new `evidence_tier` field from `bds.list_supported_formats()` and verifies that it is visible through the public API before testing the files.

In [2]:
formats = bds.list_supported_formats()
print("Supported format metadata:")
for item in formats:
    print(
        f"- {item['cycler']:9s} support={item['support_tier']:<15s} "
        f"evidence={item.get('evidence_tier')} extensions={','.join(item['extensions'])}"
    )

print("\nEvidence-tier counts:")
counts: dict[str, int] = {}
for item in formats:
    counts[item.get("evidence_tier", "missing")] = counts.get(item.get("evidence_tier", "missing"), 0) + 1
print(json.dumps(counts, indent=2))

Supported format metadata:
- neware    support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt,.tsv,.xlsx,.xls,.mat,.parquet
- arbin     support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt,.tsv,.xlsx,.xls,.mat,.parquet
- maccor    support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt,.tsv,.xlsx,.xls,.mat,.parquet
- biologic  support=fixture-backed  evidence=public-fixture-backed extensions=.mpt,.mpr,.txt,.csv
- repower   support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt
- pec       support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt
- novonix   support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt,.tsv,.xlsx,.xls,.mat,.parquet
- basytec   support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt,.tsv,.xlsx,.xls,.mat,.parquet
- landt     support=fixture-backed  evidence=public-fixture-backed extensions=.csv,.txt,.tsv,.xlsx,.xls,.mat,

## Real Dataset Inventory

These are the real datasets requested for this version check. Paths are resolved relative to the repository `tests/` directory.

In [3]:
DATASETS = [
    "HPPC_MultiSine_F10.mat",
    "HPPC_MultiSine_F11.mat",
    "INR21700_M50T_T23_Aging_EIS_SOC20_W8_Channel_4.xlsx",
    "INR21700_M50T_T23_Aging_EIS_SOC50_V4_Channel_1.xlsx",
    "INR21700_M50T_T23_Aging_HPPC_G1_Channel_3.xlsx",
    "INR21700_M50T_T23_Aging_HPPC_W8_Channel_4.xlsx",
    "INR21700_M50T_T23_Aging_UDDS_SOC20_80_CC_0_25C_N25_V4_Channel_1.1.xlsx",
    "M2_NCA_Unaged_R1_T10.xlsx",
    "OCVDis_F10.mat",
]

data_paths = [repo_root / "tests" / name for name in DATASETS]
for path in data_paths:
    print(path.name)
    print("  exists:", path.exists())
    print("  size_MB:", round(path.stat().st_size / 1_000_000, 3) if path.exists() else None)
    print("  suffix:", path.suffix)

HPPC_MultiSine_F10.mat
  exists: True
  size_MB: 0.515
  suffix: .mat
HPPC_MultiSine_F11.mat
  exists: True
  size_MB: 0.511
  suffix: .mat
INR21700_M50T_T23_Aging_EIS_SOC20_W8_Channel_4.xlsx
  exists: True
  size_MB: 0.092
  suffix: .xlsx
INR21700_M50T_T23_Aging_EIS_SOC50_V4_Channel_1.xlsx
  exists: True
  size_MB: 0.114
  suffix: .xlsx
INR21700_M50T_T23_Aging_HPPC_G1_Channel_3.xlsx
  exists: True
  size_MB: 12.89
  suffix: .xlsx
INR21700_M50T_T23_Aging_HPPC_W8_Channel_4.xlsx
  exists: True
  size_MB: 15.63
  suffix: .xlsx
INR21700_M50T_T23_Aging_UDDS_SOC20_80_CC_0_25C_N25_V4_Channel_1.1.xlsx
  exists: True
  size_MB: 51.637
  suffix: .xlsx
M2_NCA_Unaged_R1_T10.xlsx
  exists: True
  size_MB: 11.406
  suffix: .xlsx
OCVDis_F10.mat
  exists: True
  size_MB: 0.093
  suffix: .mat


## Helper Functions

The helper below runs each file through:

1. `detect_kind()` and `detect()` for routing and adapter candidates.
2. `doctor()` for problem-oriented diagnostics and fixture guidance.
3. `read_with_report()` for conversion metadata, validation, provenance, and the new temperature/current/step diagnostics.

Each file is isolated with `try/except` so one failure does not hide the rest.

In [4]:
def compact(value: Any, *, limit: int = 4000) -> str:
    text = json.dumps(value, indent=2, ensure_ascii=False, default=str)
    if len(text) > limit:
        return text[:limit] + "\n... <truncated>"
    return text


def issue_counts(issues: list[Any]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for issue in issues:
        code = issue.get("code") if isinstance(issue, dict) else getattr(issue, "code", None)
        counts[str(code)] = counts.get(str(code), 0) + 1
    return counts


def provenance_subset(provenance: list[Any]) -> list[dict[str, Any]]:
    keep = {
        "test_time_s",
        "voltage_v",
        "current_a",
        "cycle_index",
        "step_index",
        "step_time_s",
        "ambient_temperature_deg_c",
        "temperature_t1_deg_c",
        "charge_capacity_ah",
        "discharge_capacity_ah",
    }
    rows = []
    for item in provenance:
        if item.column in keep:
            rows.append(item.to_dict())
    return rows


def print_section(title: str) -> None:
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)


def run_real_dataset(path: Path) -> dict[str, Any]:
    print_section(path.name)
    result: dict[str, Any] = {"file": path.name, "path": str(path), "exists": path.exists()}
    if not path.exists():
        print("MISSING FILE")
        result["status"] = "missing"
        return result

    started = time.perf_counter()
    try:
        kind = bds.detect_kind(path)
        detection = bds.detect(path)
        result["detect_kind"] = kind.to_dict()
        result["detect"] = detection.to_dict()
        print("[detect_kind]")
        print(compact(kind.to_dict(), limit=2500))
        print("\n[detect candidates: top 6]")
        for candidate in detection.candidates[:6]:
            print("-", candidate)

        t0 = time.perf_counter()
        doctor_report = doctor(
            path,
            cycler="auto",
            current_sign="preserve",
            current_sign_check="adjacent",
            repair_policy="warn",
            detection_threshold=0.1,
        )
        result["doctor"] = doctor_report.to_dict()
        print("\n[doctor text]")
        print(doctor_report.to_text())
        print("doctor_elapsed_s:", round(time.perf_counter() - t0, 3))

        t1 = time.perf_counter()
        df, report = bds.read_with_report(
            path,
            cycler="auto",
            current_sign="preserve",
            current_sign_check="adjacent",
            repair_policy="warn",
            strict=False,
        )
        result["conversion_status"] = "ok"
        result["rows"] = df.height
        result["columns"] = list(df.columns)
        result["validation_valid"] = report.validation.valid
        result["validation_issue_counts"] = issue_counts([issue.to_dict() for issue in report.validation.issues])
        result["warnings"] = report.warnings
        result["metadata_focus"] = {
            "support_tier": report.support_tier,
            "evidence_tier": report.evidence_tier,
            "detection_confidence": report.detection_confidence,
            "current_sign_confidence": report.metadata.get("current_sign_confidence"),
            "current_sign_sanity": report.metadata.get("current_sign_sanity"),
            "temperature_semantics_confidence": report.metadata.get("temperature_semantics_confidence"),
            "temperature_semantics": report.metadata.get("temperature_semantics"),
            "semantic_sources": report.metadata.get("semantic_sources"),
            "step_cycle_semantics": report.metadata.get("step_cycle_semantics"),
            "time_sampling": report.metadata.get("time_sampling"),
            "repair_operations": report.repair_operations,
            "unmapped_columns_first_30": report.unmapped_columns[:30],
        }

        print("\n[conversion report summary]")
        print("cycler:", report.cycler)
        print("rows:", df.height)
        print("columns:", list(df.columns))
        print("validation_valid:", report.validation.valid)
        print("validation_issue_counts:", result["validation_issue_counts"])
        print("support_tier:", report.support_tier)
        print("evidence_tier:", report.evidence_tier)
        print("detection_confidence:", report.detection_confidence)
        print("warnings_count:", len(report.warnings))
        for warning in report.warnings[:12]:
            print("  warning:", warning)

        print("\n[metadata: trust diagnostics]")
        print(compact(result["metadata_focus"], limit=9000))

        print("\n[provenance subset]")
        print(compact(provenance_subset(report.provenance), limit=6000))

        print("\n[data preview: head 3]")
        print(df.head(3))
        print("\n[data preview: tail 3]")
        print(df.tail(3))
        print("read_with_report_elapsed_s:", round(time.perf_counter() - t1, 3))

    except Exception as exc:
        result["conversion_status"] = "error"
        result["error_type"] = type(exc).__name__
        result["error"] = str(exc)
        result["traceback"] = traceback.format_exc()
        print("ERROR:", type(exc).__name__, str(exc))
        print(result["traceback"])

    result["elapsed_s"] = round(time.perf_counter() - started, 3)
    print("total_elapsed_s:", result["elapsed_s"])
    return result

## Run Real-Data Diagnostics

This cell may take several minutes because several files are large Excel workbooks. It prints detailed output immediately after each file finishes.

In [5]:
all_results = []
notebook_started = time.perf_counter()
for dataset_path in data_paths:
    all_results.append(run_real_dataset(dataset_path))
print_section("ALL DATASETS SUMMARY")
print("total_notebook_dataset_elapsed_s:", round(time.perf_counter() - notebook_started, 3))
summary_rows = []
for item in all_results:
    doctor_status = (item.get("doctor") or {}).get("status")
    metadata = item.get("metadata_focus") or {}
    summary_rows.append(
        {
            "file": item.get("file"),
            "conversion_status": item.get("conversion_status"),
            "doctor_status": doctor_status,
            "rows": item.get("rows"),
            "validation_valid": item.get("validation_valid"),
            "evidence_tier": metadata.get("evidence_tier"),
            "temperature_confidence": metadata.get("temperature_semantics_confidence"),
            "current_sign_confidence": metadata.get("current_sign_confidence"),
            "elapsed_s": item.get("elapsed_s"),
            "error_type": item.get("error_type"),
        }
    )
print(compact(summary_rows, limit=12000))


HPPC_MultiSine_F10.mat
[detect_kind]
{
  "kind": "timeseries",
  "confidence": 0.8,
  "reason": "time, voltage, and current-like columns detected",
  "path": "C:\\work\\Batterydatastandard\\tests\\HPPC_MultiSine_F10.mat",
  "evidence": {
    "columns": [
      "Time",
      "Voltage",
      "Current",
      "Cycle",
      "Step",
      "Temperature"
    ]
  }
}

[detect candidates: top 6]
- {'cycler': 'generic', 'confidence': 0.4, 'reason': 'generic column-token match'}
- {'cycler': 'neware', 'confidence': 0.0, 'reason': 'no NEWARE signature'}
- {'cycler': 'arbin', 'confidence': 0.0, 'reason': 'extension fallback'}
- {'cycler': 'maccor', 'confidence': 0.0, 'reason': 'extension fallback'}
- {'cycler': 'biologic', 'confidence': 0.0, 'reason': 'extension fallback'}
- {'cycler': 'repower', 'confidence': 0.0, 'reason': 'extension fallback'}

[doctor text]
BDS doctor
Input: C:\work\Batterydatastandard\tests\HPPC_MultiSine_F10.mat
Status: ok
Data kind: timeseries (0.8)
Adapter: generic (0.4)

## Focus: New Feature Checks

This final cell extracts only the new-version fields so they are easy to inspect quickly.

In [6]:
print_section("NEW FEATURE FIELD CHECK")
for item in all_results:
    print("\nFILE:", item.get("file"))
    doctor_payload = item.get("doctor") or {}
    metadata = item.get("metadata_focus") or {}
    print("doctor.status:", doctor_payload.get("status"))
    print("doctor.suggestions:")
    for suggestion in (doctor_payload.get("suggestions") or [])[:5]:
        print("  -", suggestion)
    print("doctor.fixture_checklist_first_3:")
    for checklist_item in (doctor_payload.get("fixture_checklist") or [])[:3]:
        print("  -", checklist_item)
    print("evidence_tier:", metadata.get("evidence_tier"))
    print("temperature_semantics_confidence:", metadata.get("temperature_semantics_confidence"))
    print("temperature_semantics:", compact(metadata.get("temperature_semantics"), limit=2500))
    print("current_sign_sanity.status:", (metadata.get("current_sign_sanity") or {}).get("status"))
    print("step_cycle repeated/discontinuities:", {
        "repeated_step_segments": (metadata.get("step_cycle_semantics") or {}).get("repeated_step_segments"),
        "step_transition_discontinuities": (metadata.get("step_cycle_semantics") or {}).get("step_transition_discontinuities"),
        "inferred_fields": (metadata.get("step_cycle_semantics") or {}).get("inferred_fields"),
    })


NEW FEATURE FIELD CHECK

FILE: HPPC_MultiSine_F10.mat
doctor.status: ok
doctor.suggestions:
  - Convert with bds convert C:\work\Batterydatastandard\tests\HPPC_MultiSine_F10.mat <output.csv> --cycler generic.
doctor.fixture_checklist_first_3:
  - Keep the original header rows, column names, units, and sheet name.
  - Keep 10-20 representative data rows around the failing region.
  - Preserve time, voltage, current, status/step type, cycle, step, capacity, energy, and temperature columns when present.
evidence_tier: unit-test-backed
temperature_semantics_confidence: low
temperature_semantics: {
  "status": "ambiguous",
  "confidence": "low",
  "reason": "One or more temperature columns were mapped as ambient/chamber temperature, but the source header is semantically ambiguous.",
  "examples": [
    {
      "mapped_column": "ambient_temperature_deg_c",
      "source": "Temperature",
      "reason": "raw temperature header does not distinguish ambient/chamber from surface sensor"
    }
 